[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/573phn/dh_ad_a3/blob/main/a3.ipynb)

# Installing required libraries

In [1]:
!pip install ollama

In [2]:
!sudo apt update && sudo apt install pciutils lshw

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,480 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,972 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,6

In [3]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 54 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (427 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122387 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


### Installing Ollama

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### Starting Ollama server and saving its output to a log file ollama.log

In [5]:
!nohup ollama serve > ollama.log 2>&1 &

### Download the gemma3 model

In [11]:
!ollama pull gemma3

### List downloaded models
Please note, the `!ollama pull gemma3` command used in the previous cell does not always work on the first run. If no model is listed when `!ollama list` is used below, run the previous cell again.

In [12]:
!ollama list

NAME             ID              SIZE      MODIFIED      
gemma3:latest    a2af6cc3eb7f    3.3 GB    2 seconds ago    


# Assignment 3 - Part 2

In [13]:
import ollama
import pandas as pd
from sklearn.metrics import classification_report

In [ ]:
def prompt_ollama(lyrics: str, train_df: pd.DataFrame, shots: int = 0):
    """ Function that sends prompt to ollama model
    lyrics: the lyrics we want to predict the genre for
    train_df: Pandas DataFrame with training data
    shots: number of examples to include in the prompt, 0 by default
    """
    genres_str = ""
    # if we don't use 0-shot, add examples and possible genres to prompt
    examples = []
    if shots > 0:
        for _, row in train_df.sample(n=shots, random_state=42).iterrows():
            examples.append([{"role": "user", "content": f"Lyrics: {row.lyrics}"}, {"role": "assistant", "content": row.genre}])
        genres_str = f" You are only allowed to pick one genre from this list: {train_df.genre.unique()}"

    # send prompt to model and return a stripped result, because the model sometimes adds newlines before/after its response
    result = ollama.chat(
        model="gemma3",
        messages=[{"role": "system", "content": f"You are an expert on music, song lyrics, and music genres. Your task is to determine the genre of a song based on its lyrics. Only return the name of one genre. Do not provide any additional information.{genres_str}"}] + examples + [{"role": "user", "content": f"Lyrics: {lyrics}"}])

    return result["message"]["content"].strip()

In [15]:
# read training and test data into Pandas DataFrame
train_df = pd.read_csv("./data/genreLyrics_train.csv", sep="\t")
test_df = pd.read_csv("./data/genreLyrics_test.csv", sep="\t")

# retrieve genres from training data
genres = train_df.genre.unique()

# we'll be using few-shot (2) prompting later on, we're using random_state to always use the same examples for our prompt and print them so we can see which examples were used
print(train_df.sample(n=2, random_state=42))

# prepare lists to store the results
zero_shot_results = []
few_shot_results = []

# iterate through test data
for _, row in test_df.iterrows():
    # prompt model for each lyric and store result (zero-shot)
    zero_shot_results.append(prompt_ollama(row.lyrics, train_df))
    # repeat the same, but use few-shot this time
    few_shot_results.append(prompt_ollama(row.lyrics, train_df, shots=2))

      Unnamed: 0  genre                                             lyrics
1501       89467  Indie  When I first touched your skin\nThat was when ...
2586      182276   Rock  A black eyed dog he called at my door\nThe bla...


In [16]:
# print classification reports to get precision, recall, and f1-scores
# we're setting zero_division to 0 because we're expecting genre labels that are not in our predefined list of genres
print("Zero-shot")
print(classification_report(test_df.genre.to_list(), zero_shot_results, labels=genres, zero_division=0))

print("Few-shot")
print(classification_report(test_df.genre.to_list(), few_shot_results, labels=genres, zero_division=0))

Zero-shot
              precision    recall  f1-score   support

        Rock       0.75      0.04      0.07       655
     Country       0.36      0.45      0.40        89
  Electronic       0.00      0.00      0.00        57
         Pop       0.38      0.32      0.35       255
     Hip-Hop       0.45      0.03      0.06       163
       Metal       0.65      0.19      0.29       151
       Indie       0.00      0.00      0.00        16
         R&B       0.00      0.00      0.00        23
        Folk       0.14      0.12      0.13        17
        Jazz       0.40      0.09      0.15        43
       Other       0.00      0.00      0.00        31

   micro avg       0.40      0.12      0.19      1500
   macro avg       0.29      0.11      0.13      1500
weighted avg       0.54      0.12      0.15      1500

Few-shot
              precision    recall  f1-score   support

        Rock       0.62      0.36      0.46       655
     Country       0.31      0.49      0.38        89
  Ele

In [ ]:
# save results to csv file
combined_results_df = pd.DataFrame({"zero-shot": zero_shot_results, "few-shot": few_shot_results, "actual": test_df.genre.to_list()})
combined_results_df.to_csv("results.csv")